# 05 · Training and evaluation
**Goals:** connect batches, forward passes, losses, gradients, optimizer steps, and validation. This is the explicit equivalent of much of what an estimator's `fit` hides. Each iteration updates parameters once; an epoch visits the full training loader.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Small examples run efficiently on a laptop.
root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent
data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."
frame = pd.read_csv(data_path)
features = ["feature_1", "feature_2", "feature_3"]
train = frame[frame["split"] == "train"]
validation = frame[frame["split"] == "validation"]
test = frame[frame["split"] == "test"]
scaler = StandardScaler().fit(train[features])  # Never fit on validation or test.

def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)
x_test, y_test = tensors(test)


The loop below clears gradients, computes the forward pass and loss, computes gradients with `backward`, and updates with Adam. Evaluation uses `eval()` and `no_grad()` for distinct reasons. Mean losses are weighted by batch size, so a smaller final batch has the correct contribution.

In [ ]:
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64, shuffle=False)
model = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()  # Inputs are raw logits; labels are integer indices.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            logits = model(x_batch)
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            count += len(y_batch)
    return total_loss / count, correct / count

history = []
for epoch in range(1, 21):
    model.train()
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
    train_loss, train_accuracy = evaluate(model, DataLoader(TensorDataset(x_train, y_train), batch_size=64))
    val_loss, val_accuracy = evaluate(model, val_loader)
    history.append((epoch, train_loss, val_loss, train_accuracy, val_accuracy))
print(f"Epoch {epoch}: train loss={train_loss:.3f}, validation loss={val_loss:.3f}, validation accuracy={val_accuracy:.3f}")


In [ ]:
import matplotlib.pyplot as plt
history_frame = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_accuracy", "val_accuracy"])
history_frame.plot(x="epoch", y=["train_loss", "val_loss"], ylabel="Cross-entropy loss", title="Learning curves")
plt.show()
print(history_frame.tail())


During development, compare training and validation learning curves and change hyperparameters using validation results. The following test evaluation is a demonstration: reserve the test set until you have completed model selection in an actual project.

In [ ]:
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=64, shuffle=False)
test_loss, test_accuracy = evaluate(model, test_loader)
print(f"Held-out test: loss={test_loss:.3f}, accuracy={test_accuracy:.3f}")


## Try it
1. Replace Adam with `torch.optim.SGD` and try learning rates. Compare the curves.
2. Print gradient norms before `optimizer.step()`. What happens if `zero_grad()` is omitted?
3. Change batch size and count updates per epoch. Why is an epoch different from a step?
4. Add a confusion matrix; explain what accuracy alone can hide.